# Milestone 2: energy logging and reaction site comparisonThis version adds two things the first pass did not have.1. Potential energy capture at every recorded frame, alongside distance. UMA computes this at every step already, we just were not saving it. This lets us build an actual reaction coordinate diagram, the standard way computational chemists visualize a reaction.2. A `target_site` parameter, so the same pipeline can attack C4, C5, or C8. Literature says C4 and C5 addition are close to barrierless, while C8 (the site that leads to 8-oxoguanine) has a real, measurable barrier. That is a genuine, checkable comparison.Before running: Runtime > Change runtime type > GPU (T4 is fine).

In [ ]:
!pip install -q rdkit ase huggingface_hub fairchem-core

In [ ]:
from google.colab import userdatafrom huggingface_hub import loginlogin(token=userdata.get('HF_TOKEN'))

In [ ]:
import timeimport jsonimport numpy as npfrom rdkit import Chemfrom rdkit.Chem import AllChemfrom ase import Atomsfrom ase import unitsfrom ase.md.langevin import Langevinfrom ase.md.velocitydistribution import MaxwellBoltzmannDistributionfrom ase.constraints import Hookeanfrom ase.io import write as ase_writefrom fairchem.core import FAIRChemCalculatorfrom fairchem.core.calculate import pretrained_mlipimport torchprint("CUDA available:", torch.cuda.is_available())if not torch.cuda.is_available():    print("No GPU detected. Runtime > Change runtime type > GPU, then rerun this cell.")

In [ ]:
# Atom indices, confirmed earlier by inspecting RDKit neighbor lists on this# exact SMILES/embedding. C4 and C5 are the six membered ring fusion carbons# (site of the barrierless addition in the literature). C8 is the five# membered ring carbon that leads to 8-oxoguanine, and carries a real barrier.TARGET_SITES = {"C8": 5, "C4": 3, "C5": 7}BOND_CUTOFF = 1.7GUANINE_SMILES = "Nc1nc2[nH]cnc2c(=O)[nH]1"def build_guanine_OH_complex(target_site="C8", approach_dist=2.8, seed=42):    """Guanine plus hydroxyl radical, oriented for pi-face attack at the    chosen ring carbon. Approach geometry is a reasonable pi-face approximation,    not a located transition state, worth stating plainly if this goes in a    writeup."""    mol = Chem.MolFromSmiles(GUANINE_SMILES)    mol = Chem.AddHs(mol)    params = AllChem.ETKDGv3()    params.randomSeed = seed    AllChem.EmbedMolecule(mol, params)    AllChem.MMFFOptimizeMolecule(mol)    conf = mol.GetConformer()    site_idx = TARGET_SITES[target_site]    site_pos = np.array(conf.GetAtomPosition(site_idx))    ring_atoms = [1, 3, 5]    p0, p1, p2 = [np.array(conf.GetAtomPosition(i)) for i in ring_atoms]    normal = np.cross(p1 - p0, p2 - p0)    normal = normal / np.linalg.norm(normal)    O_pos = site_pos + normal * approach_dist    H_pos = O_pos + normal * 0.97    symbols = [atom.GetSymbol() for atom in mol.GetAtoms()] + ["O", "H"]    positions = [list(conf.GetAtomPosition(i)) for i in range(mol.GetNumAtoms())]    positions += [list(O_pos), list(H_pos)]    atoms = Atoms(symbols=symbols, positions=positions)    atoms.info["charge"] = 0    atoms.info["spin"] = 2  # doublet, one unpaired electron from the radical    radical_o_idx = len(symbols) - 2    return atoms, site_idx, radical_o_idx

In [ ]:
def run_and_save_trajectory(calc, target_site="C8", temp_k=350, timestep_fs=0.5,                             steps=2000, seed=0, approach_dist=2.8,                             cage_distance=2.2, cage_k=1.0,                             save_prefix="reaction"):    """Runs one trajectory, saves full atomic positions (xyz) and a    sidecar json with time, distance, and potential energy per frame.    Both files download automatically when done."""    atoms, site_idx, o_idx = build_guanine_OH_complex(        target_site=target_site, approach_dist=approach_dist, seed=seed)    atoms.calc = calc    atoms.set_constraint(Hookean(a1=o_idx, a2=site_idx, rt=cage_distance, k=cage_k))    MaxwellBoltzmannDistribution(atoms, temperature_K=temp_k, rng=np.random.RandomState(seed))    dyn = Langevin(atoms, timestep=timestep_fs * units.fs,                    temperature_K=temp_k, friction=0.01 / units.fs)    frames, distances, energies, times_fs = [], [], [], []    t_start = time.time()    def record():        d = atoms.get_distance(o_idx, site_idx, mic=False)        e = atoms.get_potential_energy()  # already computed this step, no extra cost        distances.append(d)        energies.append(e)        times_fs.append(len(frames) * 5 * timestep_fs)        frames.append(atoms.copy())        if len(frames) % 20 == 0:            elapsed = time.time() - t_start            print(f"    frame {len(frames)}  |  {target_site}...O = {d:.2f} A"                  f"  |  E = {e:.3f} eV  |  {elapsed:.0f}s elapsed")    dyn.attach(record, interval=5)    dyn.run(steps=steps)    xyz_path = f"{save_prefix}_{target_site}_seed{seed}.xyz"    json_path = f"{save_prefix}_{target_site}_seed{seed}_kinetics.json"    ase_write(xyz_path, frames)    with open(json_path, "w") as f:        json.dump({            "target_site": target_site,            "site_atom_index": site_idx,            "radical_o_index": o_idx,            "cage_distance": cage_distance,            "temp_k": temp_k,            "time_fs": times_fs,            "distance_A": distances,            "energy_eV": energies,        }, f)    print(f"\nSaved {len(frames)} frames to {xyz_path}")    print(f"Saved kinetics (time, distance, energy) to {json_path}")    print(f"Final distance: {distances[-1]:.2f} A  |  Final energy: {energies[-1]:.3f} eV")    print(f"Min energy: {min(energies):.3f} eV at frame {energies.index(min(energies))}")    print(f"Max energy: {max(energies):.3f} eV at frame {energies.index(max(energies))}")    from google.colab import files as colab_files    colab_files.download(xyz_path)    colab_files.download(json_path)    return frames, distances, energies, times_fs

In [ ]:
def run_swarm(target_site="C8", n_trajectories=5, temp_k=350,              approach_dist=2.8, cage_distance=2.2, cage_k=1.0, steps=2000):    """Reactive fraction sweep, now site-aware. Use this to compare how    readily each site reacts under the SAME cage and temperature settings,    the actual test of the literature's barrier ordering."""    print(f"Loading UMA model, target site = {target_site}...")    predictor = pretrained_mlip.get_predict_unit("uma-s-1p1", device="cuda")    calc = FAIRChemCalculator(predictor, task_name="omol")    results = []    for i in range(n_trajectories):        print(f"\nTrajectory {i+1}/{n_trajectories} (seed={i}, site={target_site})...")        atoms, site_idx, o_idx = build_guanine_OH_complex(            target_site=target_site, approach_dist=approach_dist, seed=i)        atoms.calc = calc        atoms.set_constraint(Hookean(a1=o_idx, a2=site_idx, rt=cage_distance, k=cage_k))        MaxwellBoltzmannDistribution(atoms, temperature_K=temp_k, rng=np.random.RandomState(i))        dyn = Langevin(atoms, timestep=0.5 * units.fs, temperature_K=temp_k, friction=0.01 / units.fs)        distances = []        def record():            distances.append(atoms.get_distance(o_idx, site_idx, mic=False))        dyn.attach(record, interval=5)        t0 = time.time()        dyn.run(steps=steps)        elapsed = time.time() - t0        tail = distances[-10:]        reacted = all(d < BOND_CUTOFF for d in tail)        print(f"  Reacted: {reacted}  |  final distance: {distances[-1]:.2f} A  |  took {elapsed:.1f}s")        results.append({"seed": i, "reacted": reacted, "final_distance": distances[-1]})    n_reacted = sum(r["reacted"] for r in results)    print(f"\n{'='*55}")    print(f"{target_site}: reactive fraction at {temp_k}K, cage={cage_distance}A: "          f"{n_reacted}/{n_trajectories} ({100*n_reacted/n_trajectories:.0f}%)")    print(f"{'='*55}")    return results

## Step A, find a cage distance that gives a real, mixed fraction for C8Try 2.4, then 2.5, adjust from there. The goal is something between 0% and 100%, not another guaranteed result.

In [ ]:
results_c8 = run_swarm(target_site="C8", n_trajectories=5, temp_k=350, cage_distance=2.4)

## Step B, the actual comparison, C4 and C5 against C8, same settingsThis is the centerpiece result. Literature predicts C4 and C5 should react far more easily than C8 under identical conditions. Run all three at whatever cage distance Step A settled on.

In [ ]:
CAGE = 2.4  # update to whatever Step A foundresults_c4 = run_swarm(target_site="C4", n_trajectories=5, temp_k=350, cage_distance=CAGE)

In [ ]:
results_c5 = run_swarm(target_site="C5", n_trajectories=5, temp_k=350, cage_distance=CAGE)

In [ ]:
results_c8_same_cage = run_swarm(target_site="C8", n_trajectories=5, temp_k=350, cage_distance=CAGE)print("\nSUMMARY")for site, res in [("C4", results_c4), ("C5", results_c5), ("C8", results_c8_same_cage)]:    n = sum(r["reacted"] for r in res)    print(f"  {site}: {n}/{len(res)} reacted")

## Step C, capture one full trajectory per site, with energy loggingOnce you know the sites react, capture real footage plus kinetics for each, for the viewer.

In [ ]:
predictor = pretrained_mlip.get_predict_unit("uma-s-1p1", device="cuda")calc = FAIRChemCalculator(predictor, task_name="omol")frames_c8, dist_c8, energy_c8, t_c8 = run_and_save_trajectory(    calc, target_site="C8", seed=0, cage_distance=CAGE)

In [ ]:
frames_c4, dist_c4, energy_c4, t_c4 = run_and_save_trajectory(    calc, target_site="C4", seed=0, cage_distance=CAGE)

In [ ]:
frames_c5, dist_c5, energy_c5, t_c5 = run_and_save_trajectory(    calc, target_site="C5", seed=0, cage_distance=CAGE)